In [2]:
# %% [markdown]
# # 📱 Google Play Review Fetcher (Project 1 - Phase 1)
# Fetches real customer feedback across 5 popular apps, maps ratings to sentiment, and exports a clean CSV.

# %% [code]
# Install dependencies (safe to run multiple times)
# !pip install -q google-play-scraper pandas tqdm

from google_play_scraper import reviews, Sort
import pandas as pd
from tqdm import tqdm
import time
import random
import os

# 🔧 Configuration
APP_LIST = [
    {"id": "com.jumia.android", "name": "Jumia (E-commerce)"},
    {"id": "com.glovo", "name": "Glovo (Food Delivery)"},
    {"id": "com.ubercab", "name": "Uber (Ride-hailing)"},
    {"id": "com.booking", "name": "Booking.com (Travel)"},
    {"id": "com.amazon.mShop.android.shopping", "name": "Amazon (Retail)"}
]

REVIEWS_PER_APP = 500
LANG = "en"
COUNTRY = "us"
SORT = Sort.MOST_RELEVANT

# 📁 Paths: Notebook is in notebooks/, data goes to root/data/
DATA_DIR = os.path.join(os.getcwd(), "..", "data")
OUTPUT_PATH = os.path.join(DATA_DIR, "gplay_multi_app_reviews.csv")

# Create data folder if missing
os.makedirs(DATA_DIR, exist_ok=True)

# ⚡ Skip if already fetched (saves time & API calls)
if os.path.exists(OUTPUT_PATH):
    print(f" Data already exists: {OUTPUT_PATH}")
    print(" Delete the file to re-fetch fresh reviews.")
else:
    all_reviews = []

    print("🔍 Fetching real customer reviews across 5 apps...\n")
    for app in tqdm(APP_LIST, desc="Apps", unit="app"):
        try:
            result, _ = reviews(
                app["id"], 
                lang=LANG, 
                country=COUNTRY, 
                sort=SORT, 
                count=REVIEWS_PER_APP
            )
            for r in result:
                r["app_name"] = app["name"]
                r["app_id"] = app["id"]
            all_reviews.extend(result)
            print(f" {app['name']}: {len(result)} reviews fetched")
            time.sleep(random.uniform(1.5, 3.0))  # Respectful delay
        except Exception as e:
            print(f"⚠️ Failed to fetch {app['name']}: {e}")

    # Convert to DataFrame
    df = pd.DataFrame(all_reviews)
    if df.empty:
        raise RuntimeError("No reviews fetched. Check app IDs, network, or library version.")

    # 🏷️ Automatic Sentiment Labeling from Ratings
    def label_sentiment(score):
        if score >= 4.0: return "positive"
        elif score <= 2.0: return "negative"
        else: return "neutral"

    df["sentiment"] = df["score"].apply(label_sentiment)

    # 🧼 Basic Cleaning & Column Selection
    df_clean = df[["app_name", "userName", "content", "score", "sentiment", "at"]].copy()
    df_clean = df_clean.rename(columns={"content": "text", "at": "date"})
    df_clean = df_clean.dropna(subset=["text"])
    df_clean = df_clean[df_clean["text"].str.strip() != ""].reset_index(drop=True)

    # 💾 Export
    df_clean.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved {len(df_clean)} reviews to {OUTPUT_PATH}")
    
# 📊 Show stats (works whether fetched or loaded)
df_stats = pd.read_csv(OUTPUT_PATH) if not os.path.exists(OUTPUT_PATH) else pd.read_csv(OUTPUT_PATH)
print("📊 Sentiment Distribution:")
print(df_stats["sentiment"].value_counts())
print("\n📈 App-wise Distribution:")
print(df_stats["app_name"].value_counts())

 Data already exists: c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\notebooks\..\data\gplay_multi_app_reviews.csv
 Delete the file to re-fetch fresh reviews.
📊 Sentiment Distribution:
sentiment
negative    1812
positive     446
neutral      242
Name: count, dtype: int64

📈 App-wise Distribution:
app_name
Jumia (E-commerce)       500
Glovo (Food Delivery)    500
Uber (Ride-hailing)      500
Booking.com (Travel)     500
Amazon (Retail)          500
Name: count, dtype: int64


In [3]:
#!pip install pandas numpy scikit-learn spacy tqdm matplotlib
#!pip install google_play_scraper


In [4]:
import pandas as pd
import os

# Fix: Notebook is in notebooks/, so go up one level to reach data/
DATA_DIR = os.path.join(os.getcwd(), "..", "data")

# Load ONLY 5,000 from Yelp (fast, sufficient for DistilBERT)
yelp_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), 
                      header=None, 
                      names=["label", "text"], 
                      nrows=5000)

yelp_df["label"] = yelp_df["label"].map({1: "negative", 2: "positive"})
yelp_df["source"] = "yelp_public"

# Load your Google Play data (2,500 reviews)
gplay_df = pd.read_csv(os.path.join(DATA_DIR, "gplay_multi_app_reviews.csv"))
gplay_df = gplay_df.rename(columns={"sentiment": "label", "content": "text"})
gplay_df["source"] = "google_play"

# Merge
df = pd.concat([yelp_df, gplay_df], ignore_index=True)
df = df.rename(columns={"label": "sentiment", "text": "review"})

print(f"Total: {len(df)} reviews")
print(df["sentiment"].value_counts())
print(f"Sources: {df['source'].value_counts().to_dict()}")

Total: 7500 reviews
sentiment
negative    4424
positive    2834
neutral      242
Name: count, dtype: int64
Sources: {'yelp_public': 5000, 'google_play': 2500}


In [5]:
print(df.head)

<bound method NDFrame.head of      sentiment                                             review  \
0     negative  Unfortunately, the frustration of being Dr. Go...   
1     positive  Been going to Dr. Goldberg for over 10 years. ...   
2     negative  I don't know what Dr. Goldberg was like before...   
3     negative  I'm writing this review to give you a heads up...   
4     positive  All the food is great here. But the best thing...   
...        ...                                                ...   
7495   neutral  If I pause my activity for a short time, the a...   
7496  negative  it's a waste paying Amazon prime when items ar...   
7497  negative  As of the recent update, my Amazon gift card i...   
7498  negative  having trouble trying to cancel a 2 monthly or...   
7499  positive  Amazon shopping batabase at your finger-tips, ...   

           source         app_name          userName  score  \
0     yelp_public              NaN               NaN    NaN   
1     yelp_publ

In [6]:
# Nettoyage des Textes
# Normalisation, suppression des doublons, mots vides et caractères parasites.

# %% [code]
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tqdm import tqdm

# Télécharger les ressources NLTK (exécution unique)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def normalize_text(text):
    """Lowercase + suppression URLs, mentions, hashtags, caractères non-alphabétiques"""
    if pd.isna(text): return ""
    t = str(text).lower()
    t = re.sub(r'http\S+|www\S+|@\S+|#\S+|[^a-z\s]', '', t)
    return re.sub(r'\s+', ' ', t).strip()

print("🧹 1/3 Normalisation...")
df['clean_review'] = df['review'].apply(normalize_text)

print("🧹 2/3 Suppression des doublons...")
before = len(df)
df = df.drop_duplicates(subset=['clean_review']).reset_index(drop=True)
print(f"   → {before - len(df)} doublons retirés.")

print(" 3/3 Tokenisation + suppression des stopwords...")
tqdm.pandas()
df['tokens'] = df['clean_review'].progress_apply(
    lambda x: [w for w in word_tokenize(x) if w not in stop_words and len(w) > 2]
)
# Rejoindre les tokens pour TF-IDF
df['final_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# Filtrer les textes vides ou trop courts après nettoyage
df = df[df['final_text'].str.len() > 5].reset_index(drop=True)
print(f" Nettoyage terminé : {len(df)} reviews prêtes.")

🧹 1/3 Normalisation...
🧹 2/3 Suppression des doublons...
   → 2 doublons retirés.
 3/3 Tokenisation + suppression des stopwords...


100%|██████████| 7498/7498 [00:05<00:00, 1428.27it/s]


 Nettoyage terminé : 7491 reviews prêtes.


In [7]:
#  Étape 2 : Transformation & Vectorisation (TF-IDF)
# Conversion du texte en représentations numériques exploitables par les modèles.

# %% [code]
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import os

print("🔢 Vectorisation TF-IDF en cours...")
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   # Unigrammes + Bigrammes
    min_df=2,             # Ignore les termes apparaissant < 2 fois
    dtype=np.float32      # Optimisation mémoire
)

X_tfidf = tfidf.fit_transform(df['final_text'])
feature_names = tfidf.get_feature_names_out()

print(f"Matrice TF-IDF : {X_tfidf.shape[0]} documents × {X_tfidf.shape[1]} features")
print(f"Top 5 features : {feature_names[:5]}")

🔢 Vectorisation TF-IDF en cours...
Matrice TF-IDF : 7491 documents × 5000 features
Top 5 features : ['ability' 'able' 'able find' 'able get' 'absolute']


In [8]:
# Étape 3 : Export du Livrable
# Données prêtes pour la modélisation (sauvegardées dans `data/`)

# %% [code]
DATA_DIR = os.path.join(os.getcwd(), "..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

# 1. Dataset texte nettoyé (avec métadonnées)
df.to_csv(os.path.join(DATA_DIR, "cleaned_dataset.csv"), index=False)

# 2. Matrice TF-IDF (format dense CSV pour compatibilité sklearn/pandas)
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=feature_names)
tfidf_df.to_csv(os.path.join(DATA_DIR, "tfidf_matrix.csv"), index=False)

print(" Livrable exporté avec succès :")
print(f"   • data/cleaned_dataset.csv  → {len(df)} lignes")
print(f"   • data/tfidf_matrix.csv     → {X_tfidf.shape[1]} colonnes")
print("Données prêtes pour la modélisation (Logistic Regression, DistilBERT, etc.)")

 Livrable exporté avec succès :
   • data/cleaned_dataset.csv  → 7491 lignes
   • data/tfidf_matrix.csv     → 5000 colonnes
Données prêtes pour la modélisation (Logistic Regression, DistilBERT, etc.)


In [9]:
# Phase 3.1 : VADER Sentiment Analysis (Baseline)
# Rule-based scoring using NLTK's VADER lexicon. Fast, interpretable, no training required.

# %% [code]
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import os
from sklearn.metrics import accuracy_score, classification_report

# Download VADER lexicon (runs once)
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

print(" Scoring 7,491 reviews with VADER...")
# Compound score: -1 (very negative) to +1 (very positive)
df['vader_compound'] = df['clean_review'].apply(lambda x: sia.polarity_scores(x)['compound'])

# Map to sentiment labels
def vader_label(compound):
    if compound >= 0.05: return 'positive'
    elif compound <= -0.05: return 'negative'
    else: return 'neutral'

df['vader_pred'] = df['vader_compound'].apply(vader_label)

# 📊 Evaluation vs Ground Truth
y_true = df['sentiment']
y_pred = df['vader_pred']

print("\n✅ VADER Baseline Results:")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}")
print("\n📝 Classification Report:")
print(classification_report(y_true, y_pred))

# 💾 Export
DATA_DIR = os.path.join(os.getcwd(), "..", "data")
df.to_csv(os.path.join(DATA_DIR, "vader_scored_dataset.csv"), index=False)
print(f"✅ Saved: vader_scored_dataset.csv")

 Scoring 7,491 reviews with VADER...

✅ VADER Baseline Results:
Accuracy: 66.69%

📝 Classification Report:
              precision    recall  f1-score   support

    negative       0.91      0.52      0.66      4416
     neutral       0.06      0.07      0.07       242
    positive       0.57      0.94      0.71      2833

    accuracy                           0.67      7491
   macro avg       0.52      0.51      0.48      7491
weighted avg       0.76      0.67      0.66      7491

✅ Saved: vader_scored_dataset.csv


In [10]:
# Phase 3.2 : DistilBERT Fine-Tuning
# Advanced sentiment classifier using Hugging Face Transformers.

# %% [code]
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Prepare Data
print("Preparing dataset for DistilBERT...")
# Map string labels to integers: negative=0, neutral=1, positive=2
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Convert to HuggingFace Dataset (faster training)
dataset = Dataset.from_pandas(df[['clean_review', 'label']])

# 2. Tokenization
model_checkpoint = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples['clean_review'], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 3. Load Model
model = DistilBertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

# 4. Training Arguments (optimized for laptop CPU)
training_args = TrainingArguments(
    output_dir="./distilbert-results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",  # ✅ Fixed: must match eval_strategy
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# 5. Evaluation Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

# 6. Trainer Setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics
)

# 7. Train Model
print("Training DistilBERT (this may take a few minutes)...")
trainer.train()

# 8. Save Final Model
model.save_pretrained("sentiment_distilbert_model")
tokenizer.save_pretrained("sentiment_distilbert_model")
print("✅ Model trained and saved to 'sentiment_distilbert_model' folder.")

Preparing dataset for DistilBERT...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1378.08it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training DistilBERT (this may take a few minutes)...


c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.351118,0.216931,0.921906,0.910091
2,0.249339,0.146114,0.958217,0.952456
3,0.204382,0.103099,0.973301,0.972084


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]
c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]
c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.68s/it]

✅ Model trained and saved to 'sentiment_distilbert_model' folder.


In [3]:
# %% [markdown]
# ## 🔍 Phase 3.2b : DistilBERT with Proper Academic Validation
# 70% Train | 15% Validation | 15% Test (Stratified, No Data Leakage)

# %% [code]
import os
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

DATA_DIR = os.path.join(os.getcwd(), "..", "data")
if os.path.exists(os.path.join(DATA_DIR, "cleaned_dataset.csv")):
    df = pd.read_csv(os.path.join(DATA_DIR, "cleaned_dataset.csv"))
    print(f"✅ Loaded cleaned dataset: {len(df)} reviews")
else:
    print("❌ cleaned_dataset.csv not found. Please run the cleaning cells first.")

print("🔪 Splitting data: 70% train, 15% validation, 15% test...")

# 1. Map labels to integers
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# 2. Stratified split (preserves class distribution across sets)
# First: carve out 15% TEST set (never seen during training)
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])
# Second: split remaining 85% into 70% train, 15% val
train_df, val_df = train_test_split(train_val_df, test_size=0.176, random_state=42, stratify=train_val_df['label'])

print(f"✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# 3. Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df[['clean_review', 'label']])
val_dataset = Dataset.from_pandas(val_df[['clean_review', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_review', 'label']])

# 4. Tokenization
model_checkpoint = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples['clean_review'], truncation=True, padding="max_length", max_length=128)

print(" Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# 5. Load Model
model = DistilBertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

# 6. Training Arguments (CPU-optimized, strict validation)
training_args = TrainingArguments(
    output_dir="./distilbert_validated_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    save_total_limit=2,  # Saves disk space
)

# 7. Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted')
    }

# 8. Trainer with SEPARATE datasets (NO LEAKAGE)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# 9. Train
print("\n Training DistilBERT with proper validation (~3 hours on CPU)...")
trainer.train()

# 10. Evaluate on HELD-OUT TEST SET
print("\n" + "="*50)
print("📊 FINAL EVALUATION ON UNSEEN TEST SET:")
print("="*50)
test_results = trainer.evaluate(tokenized_test)
print(f"Test Accuracy: {test_results['eval_accuracy']:.2%}")
print(f"Test F1 Score: {test_results['eval_f1']:.4f}")

# 11. Save validated model
model.save_pretrained("./distilbert_validated_model")
tokenizer.save_pretrained("./distilbert_validated_model")
print("\n✅ Properly validated model saved to './distilbert_validated_model'")

✅ Loaded cleaned dataset: 7491 reviews
🔪 Splitting data: 70% train, 15% validation, 15% test...
✅ Train: 5246 | Val: 1121 | Test: 1124


 Tokenizing datasets...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3556.36it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 Training DistilBERT with proper validation (~3 hours on CPU)...


c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.218682,0.519023,3,0.884342,0.878287


Test Accuracy: 88.43%
Test F1 Score: 0.8783


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.62it/s]


✅ Properly validated model saved to './distilbert_validated_model'


In [7]:
# %% [markdown]
# ## 📊 Final Test Set Evaluation (Fixed Truncation)

# %% [code]
from transformers import pipeline
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

print("📥 Loading best checkpoint...")
classifier = pipeline("text-classification", model="./distilbert_validated_model")

# Create test set
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
test_df = df[df['label'].isin(label_map.values())].sample(n=1000, random_state=42)

print(f"🔍 Evaluating on {len(test_df)} unseen reviews...")

# 🔑 FIX: Explicitly truncate to 128 tokens (matches training) to prevent >512 crash
preds = classifier(
    test_df['clean_review'].tolist(),
    max_length=128,
    truncation=True,
    batch_size=16
)

test_df['predicted'] = [p['label'].replace('LABEL_', '') for p in preds]
test_df['predicted'] = test_df['predicted'].astype(int).map({0:'negative', 1:'neutral', 2:'positive'})

print("\n" + "="*50)
print("✅ FINAL TEST SET PERFORMANCE")
print("="*50)
print(f"Accuracy: {accuracy_score(test_df['sentiment'], test_df['predicted']):.2%}\n")
print(classification_report(test_df['sentiment'], test_df['predicted']))



📥 Loading best checkpoint...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3979.31it/s]

🔍 Evaluating on 1000 unseen reviews...



✅ FINAL TEST SET PERFORMANCE
Accuracy: 94.70%

              precision    recall  f1-score   support

    negative       0.95      0.97      0.96       592
     neutral       0.63      0.46      0.53        26
    positive       0.96      0.94      0.95       382

    accuracy                           0.95      1000
   macro avg       0.85      0.79      0.81      1000
weighted avg       0.94      0.95      0.95      1000



In [8]:
# Save demo CSV with correct path
output_path = os.path.join(DATA_DIR, "final_test_results.csv")
test_df[['sentiment', 'predicted', 'clean_review']].to_csv(output_path, index=False)
print(f"\n💾 Saved to {output_path}")


💾 Saved to c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\notebooks\..\data\final_test_results.csv
